In [151]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import numpy as np
df = pd.read_csv("../data/social_posts.csv")
brands = pd.read_csv("../data/brands.csv")
july = pd.read_csv("../data/posts_july.csv")


merge-- join cols
concat-- join rows

In [152]:
merged = df.merge(brands, on="brand", how="left")

# df-	the left table
# brands - the right table
# on="brand"	the key column (it must exist in both tables)
# how=	what to do with rows that don’t match

# Like drop and rename, merge returns a new table, so store it.

"inner"	matches only	you only want rows with complete info
"left"	all left rows	the most common case: “keep all my posts and add info where it exists”
"right"	all right rows	rare. Just swap the tables and use left.
"outer"	everything	finding what’s missing on either side

4. Always check row counts

A merge can silently lose or add rows, so check .shape before and after every merge:

python
df.shape                                          # (300, 11)
df.merge(brands, on="brand", how="inner").shape  # (234, 15) ← 66 rows lost!
df.merge(brands, on="brand", how="left").shape   # (300, 15) ← all kept

Then count the NaN in a right-side column to see how many rows didn’t match:

python
merged["category"].isna().sum()

You can also find which keys don’t match, before you merge, using isin (2.4):

python
df.loc[~df["brand"].isin(brands["brand"]), "brand"].unique()   

In [153]:
# Key columns with different names: left_on / right_on

# If the left table calls the key brand but the right calls it brand_name:

# df.merge(other, left_on="brand", right_on="brand_name", how="left")

7. The duplicate-key trap: row explosion

If a key appears twice in the right table, every matching left row gets copied once per match.

python
right_dup = pd.DataFrame({"brand": ["A", "A"], "country": ["UK", "FR"]})
left.merge(right_dup, on="brand", how="inner")
  brand  likes country
0     A     10      UK
1     A     10      FR     ← same post, twice
2     A     40      UK
3     A     40      FR

Each A row matched 2 rows, so 2 A posts × 2 matches = 4 rows. Totals and means are now wrong, and there’s no error.

Check before merging: an info table should have each key once. If it does, the number of unique keys equals the number of rows:

python
brands["brand"].nunique() == len(brands)   # True means safe

8. concat: stacking rows

posts_july.csv has the same columns as social_posts.csv, just newer posts. To stack them:

python
all_posts = pd.concat([df, july], ignore_index=True)
You pass a list of tables: [df, july].
ignore_index=True renumbers the rows 0, 1, 2, … to the end.

Without ignore_index, each table keeps its old labels, so you’d have two row 0s (one from each file). .loc[0] would then return 2 rows (2.4: .loc uses labels).

	Without ignore_index	With ignore_index=True
labels	0–299, then 0–19 again	0–319
.loc[0]	2 rows	1 row

9. Summary table + info table (groupby then merge)

A very common real-world pattern:

python
totals = df.groupby("brand")["likes"].sum().reset_index()    # brand | likes
report = totals.merge(brands, on="brand", how="left")         # add budget, category…

reset_index() (2.6) turns brand back into a normal column, so it can be used as a key.

## 2.7 Cheat sheet: merge and concat

| I want… | Write |
|---|---|
| add info from another table | `df.merge(info, on="key", how="left")` |
| only rows matching in both | `how="inner"` |
| everything from both | `how="outer"` |
| key named differently | `left_on="a", right_on="b"` |
| readable suffixes | `suffixes=("_post", "_brand")` |
| unmatched rows after merge | `merged["right_col"].isna().sum()` |
| keys with no match (before) | `df.loc[~df["key"].isin(info["key"]), "key"].unique()` |
| is the key unique? | `info["key"].nunique() == len(info)` |
| stack same-column tables | `pd.concat([a, b], ignore_index=True)` |
| groupby result → merge | `df.groupby("k")["v"].sum().reset_index().merge(info, on="k")` |

**Rules**
- Check `.shape` before and after every merge
- left = keep all my rows; inner = drop non-matches
- Duplicate keys in the info table → row explosion (no error!)
- concat needs `ignore_index=True` or you get repeated labels

In [154]:
left = pd.DataFrame({"brand": ["A", "B", "C", "A"], "likes": [10, 20, 30, 40]})
right = pd.DataFrame({"brand": ["A", "B", "D"], "country": ["UK", "US", "CA"]})
right_dup = pd.DataFrame({"brand": ["A", "A"], "country": ["UK", "FR"]})

left,right, right_dup

(  brand  likes
 0     A     10
 1     B     20
 2     C     30
 3     A     40,
   brand country
 0     A      UK
 1     B      US
 2     D      CA,
   brand country
 0     A      UK
 1     A      FR)

In [155]:
left.merge(right, on="brand", how="inner").shape #3,3
left.merge(right, on="brand", how="left").shape #4,3
left.merge(right, on="brand", how="right").shape #4,3
left.merge(right, on="brand", how="outer").shape #5,3 
left.merge(right, on="brand", how="left")["country"].isna().sum() #1
left.merge(right, on="brand", how="right")["likes"].isna().sum() #1
left.merge(right_dup, on="brand", how="left").shape #(careful: what happens to B and C?) 6,3, B,C become null for country
pd.concat([left, left]).shape #8,2
pd.concat([left, left]).index[-1] # 3 : what is the last label?
pd.concat([left, left], ignore_index=True).index[-1] #7

(3, 3)

(4, 3)

(4, 3)

(5, 3)

np.int64(1)

np.int64(1)

(6, 3)

(8, 2)

np.int64(3)

7

In [156]:
# B1. Look before you merge

# (a) Show brands.
brands
# (b) Show its shape.
brands.shape
# (c) Check that brand is unique in brands (the nunique check).
brands['brand'].nunique()==len(brands)
# (d) Find which brand(s) appear in df but not in brands.
set(df['brand'].unique()) - set(brands['brand'].unique())
df.loc[~df["brand"].isin(brands["brand"]), "brand"].unique() 
# (e) Comment: which brand is in brands but has no posts?
set(brands['brand'].unique()) - set(df['brand'].unique())
brands.loc[~brands["brand"].isin(df["brand"]), "brand"].unique() 

,brand,category,hq_country,followers,monthly_budget
0,Glowly,beauty,UK,91000,12000
1,UrbanThread,fashion,US,45500,8000
2,SnackBox,food,US,130200,25000
3,FitFuel,fitness,UK,66800,9500
4,PetPal,pets,CA,15400,3000


(5, 5)

True

{'HomeNest'}

<ArrowStringArray>
['HomeNest']
Length: 1, dtype: str

{'PetPal'}

<ArrowStringArray>
['PetPal']
Length: 1, dtype: str

In [157]:
# B2. Inner merge

# (a) Inner merge df with brands on brand. Store it as m_inner.
m_inner= df.merge(brands, on='brand', how='inner')
# (b) Show its shape.
m_inner.shape
# (c) Comment: how many rows were lost compared to df, and which brand’s posts were they?
#66 lost, lost for HomeNest because brands didn't have it

(234, 15)

In [158]:
# B3. Left merge

# (a) Left merge df with brands on brand. Store it as m_left.
m_left= df.merge(brands, on='brand', how='left')
# (b) Show its shape.
m_left.shape
# (c) Count the NaN in m_left["category"].
m_left['category'].isna().sum()
# (d) Comment: which brand are those NaN rows?
m_left[m_left['brand']=='HomeNest']
#HomeNest--66 rows, 15 cols

(300, 15)

np.int64(66)

,post_id,date,platform,brand,post_type,followers_x,likes,comments,shares,sponsored,caption_length,category,hq_country,followers_y,monthly_budget
1,P0002,2026-01-02,TikTok,HomeNest,image,27506,485,12.0,12.0,False,311,NaN,NaN,NaN,NaN
9,P0010,2026-01-07,TikTok,HomeNest,video,27195,1801,85.0,14.0,False,85,NaN,NaN,NaN,NaN
16,P0017,2026-01-10,Facebook,HomeNest,image,27818,213,12.0,5.0,False,296,NaN,NaN,NaN,NaN
23,P0024,2026-01-16,Instagram,HomeNest,video,28116,445,14.0,7.0,True,41,NaN,NaN,NaN,NaN
31,P0032,2026-01-18,Instagram,HomeNest,image,27650,700,23.0,10.0,False,375,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
281,P0282,2026-06-18,TikTok,HomeNest,image,27278,406,9.0,10.0,False,393,NaN,NaN,NaN,NaN
286,P0287,2026-06-22,Instagram,HomeNest,carousel,28290,147,4.0,2.0,True,232,NaN,NaN,NaN,NaN
291,P0292,2026-06-26,Facebook,HomeNest,video,27068,106,6.0,2.0,False,102,NaN,NaN,NaN,NaN
296,P0297,2026-06-28,Instagram,HomeNest,image,26391,188,6.0,2.0,True,137,NaN,NaN,NaN,NaN


In [159]:
# B4. Suffixes

# (a) Show m_left.columns.
m_left.columns
# (b) Comment: which columns got _x / _y, and why?
# followers >> x is from df, y is from brands, due to duplicate col names
# (c) Redo the left merge with suffixes=("_post", "_brand"). Store it as m.
m= df.merge(brands, on='brand', how= 'left', suffixes=("_post", "_brand"))
# (d) Show m.columns to confirm.
m.columns

Index(['post_id', 'date', 'platform', 'brand', 'post_type', 'followers_x',
       'likes', 'comments', 'shares', 'sponsored', 'caption_length',
       'category', 'hq_country', 'followers_y', 'monthly_budget'],
      dtype='str')

Index(['post_id', 'date', 'platform', 'brand', 'post_type', 'followers_post',
       'likes', 'comments', 'shares', 'sponsored', 'caption_length',
       'category', 'hq_country', 'followers_brand', 'monthly_budget'],
      dtype='str')

In [160]:
# B5. Outer merge

# (a) Outer merge df with brands. Show its shape.
m_outer= df.merge(brands, on='brand', how='outer')
m_outer.shape
# (b) Show the row(s) where post_id is missing.
m_outer[m_outer['post_id'].isna()]
# (c) Comment: why does this row exist?
#outer_merge ensures that the extra brand in brands is merged but it has no post data because it doesn't exist in df

(301, 15)

,post_id,date,platform,brand,post_type,followers_x,likes,comments,shares,sponsored,caption_length,category,hq_country,followers_y,monthly_budget
184,NaN,NaN,NaN,PetPal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pets,CA,15400.0,3000.0


In [161]:
# B6. Analysis with the merged table (use m)

# (a) Mean likes per category.
m.groupby('category')['likes'].mean().round(1).sort_values(ascending=False)
# (b) Round to 1.
# (c) Sort highest first.
# (d) Mean likes per hq_country, rounded to 1.
m.groupby('hq_country')['likes'].mean().round(1).sort_values(ascending=False)
# (e) Comment: HomeNest has no category. Was it included in (a)? (Hint: what does groupby do with NaN keys?)
# Nans become a category , but it's now showing in the mean table? default for pandas is dropna=T, so they're getting ignored

category
food       2688.8
beauty     1639.6
fitness    1179.4
fashion     841.9
Name: likes, dtype: float64

hq_country
US    1797.2
UK    1421.2
Name: likes, dtype: float64

In [162]:
# B7. Budget efficiency (groupby + merge)

# (a) Total likes per brand: groupby, then sum, then .reset_index(). Store it as totals.
totals= m.groupby('brand')['likes'].sum().reset_index()
# (b) Inner merge totals with brands. Store it as report.
report=totals.merge(brands, on='brand', how='inner')
# (c) Make likes_per_1k = likes / monthly_budget × 1000.
report['likes_per_1k']= np.round(report['likes']/report['monthly_budget'] *1000, 1)
# (d) Round to 1.
# (e) Show brand, likes, monthly_budget and likes_per_1k, sorted by likes_per_1k, highest first.
report[['brand', 'likes', 'monthly_budget', 'likes_per_1k']].sort_values(['likes_per_1k', 'likes'], ascending=False)
# (f) Comment: the brand with the most total likes, is it also the best value for money?
#No, snackbox has more total likes, but less value for money. Glowly has second highest likes but is first in likes/1k.

,brand,likes,monthly_budget,likes_per_1k
1,Glowly,101657,12000,8471.4
0,FitFuel,66044,9500,6952.0
2,SnackBox,161327,25000,6453.1
3,UrbanThread,47147,8000,5893.4


In [163]:
# B8. concat

# (a) Stack df and july into all_posts with ignore_index=True.
all_posts= pd.concat([df, july], ignore_index=True)
# (b) Show its shape.
all_posts.shape
# (c) Show the last index label (all_posts.index[-1]).
all_posts.index[-1]
# (d) Posts per brand in all_posts (use .size()).
all_posts.groupby('brand').size()

(320, 11)

319

brand
FitFuel        57
Glowly         64
HomeNest       75
SnackBox       64
UrbanThread    60
dtype: int64

In [164]:
# B9. The duplicate trap

# (a) Make bad_brands = brands with the Glowly row added a second time. (Hint: concat brands with brands[brands["brand"] == "Glowly"].)
glowly_dupe= brands[brands["brand"] == "Glowly"]
bad_brands= pd.concat([brands, glowly_dupe])
# (b) Run the nunique check on bad_brands.
bad_brands['brand'].nunique()== len(bad_brands)
# (c) Left merge df with bad_brands. Show its shape.
bad_brand_merge= df.merge(bad_brands, on='brand', how='left')
# (d) Comment: why is it bigger than 300? By exactly how many rows,and why that number? (Hint: how many Glowly posts are there?)
bad_brand_merge.shape
# (df['brand']=='Glowly').sum() >> 62 rows, merged twice because of the extra glowly row in bad_brands

False

(362, 15)

In [165]:
brands2 = brands.rename(columns={"brand": "brand_name"})
df.merge(brands2, left_on="brand", right_on="brand_name", how="left")
#brands2 has the brand col names as brand_name. we can either merge with brands, or change the on='brand' to 'brand_name'

,post_id,date,platform,brand,post_type,followers_x,likes,comments,shares,sponsored,caption_length,brand_name,category,hq_country,followers_y,monthly_budget
0,P0001,2026-01-01,Instagram,Glowly,image,82837,631,34.0,16.0,False,158,Glowly,beauty,UK,91000.0,12000.0
1,P0002,2026-01-02,TikTok,HomeNest,image,27506,485,12.0,12.0,False,311,NaN,NaN,NaN,NaN,NaN
2,P0003,2026-01-02,Instagram,FitFuel,video,61341,3158,88.0,52.0,False,277,FitFuel,fitness,UK,66800.0,9500.0
3,P0004,2026-01-03,Instagram,SnackBox,image,118210,1612,75.0,27.0,False,236,SnackBox,food,US,130200.0,25000.0
4,P0005,2026-01-04,TikTok,SnackBox,image,122826,1387,70.0,15.0,False,121,SnackBox,food,US,130200.0,25000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,P0296,2026-06-28,Instagram,UrbanThread,image,42560,234,7.0,6.0,False,221,UrbanThread,fashion,US,45500.0,8000.0
296,P0297,2026-06-28,Instagram,HomeNest,image,26391,188,6.0,2.0,True,137,NaN,NaN,NaN,NaN,NaN
297,P0298,2026-06-29,Instagram,HomeNest,carousel,26263,321,17.0,2.0,False,298,NaN,NaN,NaN,NaN,NaN
298,P0299,2026-06-29,TikTok,SnackBox,image,125910,4107,138.0,113.0,True,171,SnackBox,food,US,130200.0,25000.0


In [166]:
stacked = pd.concat([df, july], ignore_index=True)
stacked.loc[0]
#add ignore_index=True

post_id                P0001
date              2026-01-01
platform           Instagram
brand                 Glowly
post_type              image
followers              82837
likes                    631
comments                34.0
shares                  16.0
sponsored              False
caption_length           158
Name: 0, dtype: object

In [167]:
x = df.merge(brands, on="brand", how="left")
x.groupby("brand")["likes"].mean()
#inner join only keeps common rows. should do a left join to keep all rows from df. 

brand
FitFuel        1179.357143
Glowly         1639.629032
HomeNest        459.227273
SnackBox       2688.783333
UrbanThread     841.910714
Name: likes, dtype: float64

In [168]:
# a = pd.DataFrame({"id": [1, 2, 3, 4], "city": ["X", "Y", "Z", "X"]})
# b = pd.DataFrame({"city": ["X", "Y", "W"], "pop": [100, 200, 300]})
# a,b